# Baseline di popolarità (MINDsmall)

**Obiettivo**: implementare una baseline semplice e non personalizzata, che ordina i candidati di ogni impression in base alla **popolarità globale** (conteggio dei click), calcolata esclusivamente sul set di training.
Valutiamo poi sul DEV con metriche di ranking a livello di impression (AUC, MRR, nDCG@5/10, Precision@5, Recall@5) e calcoliamo un proxy di novelty.

## 1) Caricamento dei dati

Carichiamo:
- i **comportamenti di training**, per stimare la popolarità globale degli articoli (conteggio dei click);
- i **comportamenti di dev**, per valutare le prestazioni di ranking.

Questo evita ogni data leakage: il DEV non viene mai usato per calcolare gli score di popolarità.

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.metrics import roc_auc_score
from tqdm import tqdm  # evita i warning di IProgress

# --- 1) Caricamento dei dati ---
news_cols = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
behav_cols = ['imp_id', 'user_id', 'time', 'history', 'impressions']

train_behav = pd.read_csv('../dataset/MINDsmall_train/behaviors.tsv', sep='\t', names=behav_cols)
dev_behav   = pd.read_csv('../dataset/MINDsmall_dev/behaviors.tsv',   sep='\t', names=behav_cols)

print("Impression di training:", len(train_behav))
print("Impression di dev:", len(dev_behav))

## 2) Modello di popolarità globale

Definiamo lo score della baseline come:

**score(news) = numero di click(news) nel TRAIN**

Solo gli articoli cliccati almeno una volta nel TRAIN ricevono uno score non nullo; tutti gli altri hanno score predefinito pari a 0.

In [ ]:
# --- 2) Scoring di popolarità (solo TRAIN, per evitare leakage) ---
def get_popularity_scores(behav_df: pd.DataFrame) -> Counter:
    """
    Popolarità globale = conteggio dei click per news_id, calcolato dalle impression con etichetta '-1'.
    """
    clicked_ids = (
        behav_df["impressions"]
        .dropna()
        .astype(str)
        .str.split()
        .explode()
    )
    clicked_ids = clicked_ids[clicked_ids.str.endswith("-1")]
    clicked_ids = clicked_ids.str.split("-").str[0]
    return Counter(clicked_ids)

pop_scores = get_popularity_scores(train_behav)
total_clicks = sum(pop_scores.values())
print("Articoli unici cliccati nel TRAIN:", len(pop_scores))
print("Click totali nel TRAIN:", total_clicks)

## 3) Analisi del cold-start (candidati del DEV)

Misuriamo la frazione di articoli candidati nelle impression del DEV che hanno uno **score di popolarità pari a zero** (mai cliccati nel TRAIN).
Questo quantifica quanto spesso la baseline debba affidarsi al tie-breaking tra molti articoli con score=0.

Risultato: **quota di cold-start = 54.86%** (score=0 tra i candidati del DEV).

In [ ]:
# --- 3) Quota di cold-start tra i candidati del DEV (score==0) ---
dev_items = dev_behav["impressions"].dropna().astype(str).str.split().explode()
dev_news_ids = dev_items.str.split("-").str[0]

cold_share = (dev_news_ids.map(lambda x: pop_scores.get(x, 0) == 0).mean()) * 100
print(f"Quota di cold-start tra i candidati del DEV (pop_score==0): {cold_share:.2f}%")

## 4) Output di ranking deterministico (opzionale)

Alcune pipeline richiedono di produrre una lista di ranking rigorosa per ogni impression.
Per garantire la riproducibilità, risolviamo le parità in modo deterministico ordinando i candidati per:
1) score di popolarità (decrescente), poi
2) news_id (crescente).

Nota: questa generazione di file è indipendente dalla valutazione offline riportata di seguito.

In [ ]:
# --- 4) Output di ranking deterministico (opzionale, in stile file di submission) ---
def generate_baseline_ranks(behav_df: pd.DataFrame, scores: Counter) -> list[str]:
    """
    Formato di output: imp_id [r1 r2 ...]
    Il ranking è deterministico: ordina per (score decrescente, news_id crescente) per risolvere le parità.
    """
    lines = []
    for _, row in behav_df.iterrows():
        imp_id = row["imp_id"]
        raw = str(row["impressions"]).split()
        candidates = [c.split("-")[0] for c in raw]

        # ordinamento deterministico per score e poi per id
        order = sorted(range(len(candidates)),
                       key=lambda i: (-scores.get(candidates[i], 0), candidates[i]))

        # rank allineati all'ordine originale dei candidati (1 = migliore)
        ranks = [0] * len(candidates)
        for r, idx in enumerate(order, start=1):
            ranks[idx] = r

        lines.append(f"{imp_id} [{' '.join(map(str, ranks))}]")
    return lines

baseline_output = generate_baseline_ranks(dev_behav, pop_scores)

with open('baseline_prediction.txt', 'w') as f:
    f.write("\n".join(baseline_output))

print("baseline_prediction.txt salvato")
print("Esempio:", baseline_output[0])

## 5) Valutazione offline sul DEV (valutazione pulita)

Valutiamo gli score di popolarità grezzi **senza alcuna componente casuale**.
Questo produce metriche deterministiche e misura esclusivamente il segnale del modello (popolarità globale), non la casualità del tie-breaking.

Metriche:
- **AUC** (solo quando in un'impression sono presenti sia etichette positive sia negative),
- **MRR** (primo elemento rilevante),
- **nDCG@5 / nDCG@10**,
- **Precision@5 / Recall@5**.

In [ ]:
# --- 5) Valutazione offline pulita sul DEV (senza jitter) ---
def mrr_first_hit(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """MRR standard: reciproco del rank del primo elemento rilevante."""
    order = np.argsort(y_score)[::-1]
    y_sorted = y_true[order]
    hits = np.where(y_sorted == 1)[0]
    return 1.0 / (hits[0] + 1) if len(hits) > 0 else 0.0

def ndcg_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    """
    nDCG@K per etichette binarie.
    """
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    # Ordina le etichette
    order = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[order][:k]

    # DCG
    discounts = np.log2(np.arange(2, len(y_true_sorted) + 2))
    dcg = np.sum(y_true_sorted / discounts)

    # DCG ideale (IDCG)
    ideal_sorted = np.sort(y_true)[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal_sorted) + 2))
    idcg = np.sum(ideal_sorted / ideal_discounts)

    if idcg == 0.0:
        return 0.0

    return float(dcg / idcg)

def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int = 5) -> float:
    order = np.argsort(y_score)[::-1][:k]
    return float(np.mean(y_true[order]))

def recall_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int = 5) -> float:
    order = np.argsort(y_score)[::-1][:k]
    denom = int(np.sum(y_true))
    return float(np.sum(y_true[order]) / denom) if denom > 0 else 0.0

metrics = {"auc": [], "mrr": [], "ndcg@5": [], "ndcg@10": [], "p@5": [], "r@5": []}

print("Avvio della valutazione sul DEV...")
for _, row in tqdm(dev_behav.iterrows(), total=len(dev_behav), desc="Valutazione"):
    items = str(row["impressions"]).split()
    y_true = np.array([int(it.split("-")[1]) for it in items], dtype=int)
    candidates = [it.split("-")[0] for it in items]

    # score di popolarità grezzi (deterministici)
    y_score = np.array([pop_scores.get(c, 0) for c in candidates], dtype=float)

    n_pos = int(y_true.sum())
    # l'AUC richiede la presenza di entrambe le classi
    if 0 < n_pos < len(y_true):
        metrics["auc"].append(roc_auc_score(y_true, y_score))

    # le metriche di ranking richiedono almeno un positivo
    if n_pos > 0:
        metrics["mrr"].append(mrr_first_hit(y_true, y_score))
        metrics["ndcg@5"].append(ndcg_at_k(y_true, y_score, k=5))
        metrics["ndcg@10"].append(ndcg_at_k(y_true, y_score, k=10))
        metrics["p@5"].append(precision_at_k(y_true, y_score, k=5))
        metrics["r@5"].append(recall_at_k(y_true, y_score, k=5))

results = {k: float(np.mean(v)) if len(v) else float("nan") for k, v in metrics.items()}

print("\n" + "="*30)
print("PRESTAZIONI FINALI DELLA BASELINE (DEV)")
print("="*30)
for k, v in results.items():
    print(f"{k.upper():<7}: {v:.4f}")

### Risultati (DEV)

- **AUC** = 0.5318
- **MRR** = 0.2666
- **nDCG@5** = 0.2457
- **nDCG@10** = 0.3095
- **P@5** = 0.1065
- **R@5** = 0.3648

Interpretazione: la popolarità separa le classi solo leggermente meglio del caso casuale (AUC appena sopra 0.5) e fornisce una qualità di ranking modesta, poiché ignora completamente l'intento, il contesto e la recency dell'utente.

## 6) Novelty (self-information) sui top-5

Calcoliamo un semplice proxy di novelty per i primi 5 articoli raccomandati in ciascuna impression:
**novelty(item) = -log2(P(item))**, dove **P(item)** è stimata dalle frequenze di click nel TRAIN.

Una novelty più bassa indica raccomandazioni più comuni/popolari; una novelty più alta indica articoli più rari.
Per gli articoli mai visti (score=0), usiamo una piccola probabilità di fallback.

In [ ]:
# --- 6) Novelty (self-information) sulle prime 5 raccomandazioni ---
# P(item) dalla distribuzione dei click nel TRAIN
total_clicks = sum(pop_scores.values())
item_prob = {k: v / total_clicks for k, v in pop_scores.items()}
fallback_prob = 1.0 / (total_clicks + 1)  # per gli articoli mai visti

def topk_deterministic(candidates, scores, k=5):
    return sorted(candidates, key=lambda x: (-scores.get(x, 0), x))[:k]

novelty_scores = []
for _, row in tqdm(dev_behav.iterrows(), total=len(dev_behav), desc="Novelty"):
    candidates = [it.split("-")[0] for it in str(row["impressions"]).split()]
    top5 = topk_deterministic(candidates, pop_scores, k=5)

    session = []
    for item in top5:
        p = item_prob.get(item, fallback_prob)
        session.append(-np.log2(p))
    novelty_scores.append(float(np.mean(session)))

print(f"Novelty media (top-5): {np.mean(novelty_scores):.4f}")

## Conclusioni

Questa baseline di popolarità fornisce un limite inferiore utile:
- ottiene prestazioni modeste in termini di ranking (MRR e nDCG) e risulta solo leggermente sopra il caso casuale in AUC;
- il cold-start è sostanziale (il 54.86% dei candidati del DEV ha score=0), il che ne limita l'efficacia e rende frequente il tie-breaking;
- la novelty è relativamente bassa/moderata, come atteso per raccomandazioni guidate dalla popolarità.

Prossimi passi: introdurre personalizzazione e sensibilità alla recency tramite modellazione content-based dell'utente e segnali ibridi.